Information gain: $IG=L(R_p)-\frac{|R_1|L(R_1)+|R_2|L(R_2)}{|R_1|+|R_2|}$,\
where $L(R)=-\sum_{c}p_c\log_2 p_c$\
$p_c=\frac{|R_c|}{|R|}$


In [1]:
import numpy as np
import pandas as pd

In [2]:
class Node:
    def __init__(self,feature=None,threshold=None,left=None,right=None,*,value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value
    def IsLeaf(self):
        return self.value != None
class DecisionTree:
    def __init__(self,max_depth=50,min_samples=10,n_features=None,random_state=42):
        self.max_depth=max_depth
        self.min_samples=min_samples
        self.n_features=n_features
        self.root=None
        self.rng=np.random.RandomState(random_state)
    def fit(self,X,y):
        self.n_features=X.shape[1] if not self.n_features else min(X.shape[1],self.n_features)
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        self.root=self._grow_tree(self.X,self.y)
    def _grow_tree(self,X,y,depth=0):
        m,n=X.shape
        classes,counts=np.unique(y,return_counts=True)
        n_classes=len(classes)
        if depth>=self.max_depth or self.min_samples>len(y) or n_classes==1:
            return Node(value=classes[np.argmax(counts)])
        feature_idxs=self.rng.choice(n,self.n_features,replace=False)
        best_feature,best_thr=self._optimal_split(X,y,feature_idxs)
        l_idxs,r_idxs=self._split(X[:,best_feature],best_thr)
        left=self._grow_tree(X[l_idxs],y[l_idxs],depth+1)
        right=self._grow_tree(X[r_idxs],y[r_idxs],depth+1)
        return Node(best_feature,best_thr,left,right)
    def _optimal_split(self,X,y,feature_idxs):
        best_ig=-1
        feature_idx,threshold=None,None
        for idx in feature_idxs:
            X_col=X[:,idx]
            thresholds=np.unique(X_col)
            for thr in thresholds:
                ig=self._calculate_IG(X_col,y,thr)
                if ig>best_ig:
                    best_ig=ig
                    feature_idx=idx
                    threshold=thr
        return feature_idx,threshold
    def _entropy(self,y):
            c_sum=0
            for c in np.unique(y):
                p_c=len(y[y==c])/len(y)
                c_sum+=p_c*np.log2(p_c)
            return -c_sum
    def _calculate_IG(self,X,y,threshold):      
        parent_e=self._entropy(y)
        l_idxs,r_idxs=self._split(X,threshold)
        l_e=self._entropy(y[l_idxs])
        r_e=self._entropy(y[r_idxs])
        n_l=len(l_idxs)
        n_r=len(r_idxs)
        return parent_e-(n_l*l_e+n_r*r_e)/(n_l+n_r)
    def _split(self,X,threshold):
        return np.argwhere(X<=threshold).flatten(), np.argwhere(X>threshold).flatten()
    def _check_tree(self,x,node):
        if node.IsLeaf():
            return node.value
        if x[node.feature]<=node.threshold:
            return self._check_tree(x,node.left)
        return self._check_tree(x,node.right)
    def predict(self,X):
        return np.array([self._check_tree(x,self.root) for x in X])       

In [3]:
df=pd.read_csv('data/iris.csv')

In [4]:
X,y=df.loc[:,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:,'Species']

In [5]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [34]:
dt=DecisionTree(max_depth=3,min_samples=10)
dt.fit(X_train.values,y_train.values)
preds=dt.predict(X_test.values)

In [35]:
np.mean(preds==y_test)

np.float64(0.9666666666666667)

In [9]:
preds

array([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2,
       1, 1, 2, 2, 1, 0, 2, 0])